# Agent-to-Agent Invoice Settlement with Claude and DPX

This notebook shows two independent Claude agents settling a real payment obligation between themselves — no human in the loop, no shared credentials, no out-of-band coordination.

**The pattern:**
1. **Agent A** (the payee) creates an invoice describing what it's owed.
2. Agent A hands the `invoiceId` to Agent B through any channel — a message, a shared task queue, an API response. That's the *only* thing the two agents need to agree on.
3. **Agent B** (the payer) looks up the invoice, decides whether it's legitimate, and pays it by ID.
4. DPX runs the oracle stability check, AML/sanctions screen, and ESG scoring automatically at pay time — neither agent implements any of that itself.

This notebook runs in **sandbox mode by default** — no real funds required to follow along. At the end, we show the actual receipt from a real settlement run on Base mainnet, so you can see what the non-sandbox path produces.

## Setup

In [ ]:
%pip install anthropic httpx python-dotenv --quiet

In [ ]:
import os
import json
import httpx
import anthropic
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

SANDBOX = os.environ.get("SANDBOX", "true").lower() != "false"
AGENT_URL = "https://agent.untitledfinancial.com"

print(f"Running in {'SANDBOX' if SANDBOX else 'LIVE'} mode")

## Define the DPX tools

Three tools, one per side of the transaction. Agent A only ever sees `create_invoice`. Agent B only ever sees `get_invoice` and `pay_invoice`.

In [ ]:
creator_tools = [
    {
        "name": "create_invoice",
        "description": (
            "Create an invoice requesting payment from another agent. Returns an "
            "invoiceId to hand off to the paying agent through any channel."
        ),
        "input_schema": {
            "type": "object",
            "required": ["amount", "recipientAddress", "description"],
            "properties": {
                "amount": {"type": "number", "description": "Amount in USD"},
                "recipientAddress": {"type": "string", "description": "EVM wallet address to receive payment"},
                "description": {"type": "string"},
            },
        },
    }
]

payer_tools = [
    {
        "name": "get_invoice",
        "description": "Retrieve an invoice by ID to verify amount and description before paying.",
        "input_schema": {
            "type": "object",
            "required": ["invoiceId"],
            "properties": {"invoiceId": {"type": "string"}},
        },
    },
    {
        "name": "pay_invoice",
        "description": (
            "Pay an invoice by ID. DPX runs the oracle stability check, AML/sanctions "
            "screen, and ESG scoring automatically before authorizing."
        ),
        "input_schema": {
            "type": "object",
            "required": ["invoiceId"],
            "properties": {"invoiceId": {"type": "string"}},
        },
    },
]

## Implement the tool calls

In [ ]:
def create_invoice(amount, recipientAddress, description):
    r = httpx.post(f"{AGENT_URL}/invoice", json={
        "amount": amount, "recipientAddress": recipientAddress,
        "description": description, "currency": "USD",
    })
    return r.json()

def get_invoice(invoiceId):
    r = httpx.get(f"{AGENT_URL}/invoice/{invoiceId}")
    return r.json()

def pay_invoice(invoiceId):
    r = httpx.post(f"{AGENT_URL}/invoice/{invoiceId}/pay", json={
        "recipientAddress": None,  # invoice already carries the recipient
        "sandbox": SANDBOX,
    })
    return r.json()

creator_impl = {"create_invoice": create_invoice}
payer_impl = {"get_invoice": get_invoice, "pay_invoice": pay_invoice}

## Agent A: create the invoice

Agent A reasons about what it's owed and creates the invoice — it never touches a wallet or sees Agent B's identity.

In [ ]:
def run_agent(system_prompt, tools, tool_impl, user_message, max_turns=4):
    messages = [{"role": "user", "content": user_message}]
    for _ in range(max_turns):
        response = client.messages.create(
            model="claude-sonnet-5",
            max_tokens=1024,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason != "tool_use":
            return response, messages
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = tool_impl[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                })
        messages.append({"role": "user", "content": tool_results})
    return response, messages

RECEIVER_ADDRESS = "0x0000000000000000000000000000000000dEaD"  # replace with a real address

_, agent_a_messages = run_agent(
    system_prompt="You are Agent A. You just completed a task for another agent and are owed payment. Create an invoice for it.",
    tools=creator_tools,
    tool_impl=creator_impl,
    user_message=(
        f"You completed a data-processing job for another agent. Invoice them $1 USD, "
        f"payable to {RECEIVER_ADDRESS}, description 'Data processing services rendered'."
    ),
)

invoice_id = None
for msg in agent_a_messages:
    if msg["role"] == "user" and isinstance(msg["content"], list):
        for block in msg["content"]:
            if block.get("type") == "tool_result":
                invoice_id = json.loads(block["content"]).get("invoiceId")

print(f"Invoice created: {invoice_id}")

## Handoff

The *only* thing that crosses from Agent A to Agent B is `invoice_id` — a string. No API key, no shared secret, no wallet address exchange (the invoice already carries the recipient). In a real deployment this would arrive via a message, a task queue, or an HTTP callback.

## Agent B: verify and pay

Agent B independently decides whether the invoice looks legitimate before paying — this is its own reasoning, not something DPX does for it.

In [ ]:
response, agent_b_messages = run_agent(
    system_prompt=(
        "You are Agent B. Another agent sent you an invoice ID. Look it up, and if the "
        "amount and description look reasonable for a completed task, pay it."
    ),
    tools=payer_tools,
    tool_impl=payer_impl,
    user_message=f"You received invoice {invoice_id} from a counterparty agent. Check it and pay if it looks legitimate.",
)

print(response.content[-1].text if response.content else "(no final text — check tool results above)")

## What DPX ran automatically, that neither agent implemented

| Check | What it does |
|---|---|
| Oracle gate | Blocks settlement if global macro conditions are UNSTABLE |
| AML / sanctions screen | Checked against OFAC/EU/UN lists, both addresses |
| FATF R16 | Travel Rule compliance attestation |
| ESG score | Adjusts fee based on counterparty ESG score, 100% of ESG portion redistributed on-chain |
| AI decision | `EXECUTE` / `HOLD` / `BLOCK` with a confidence score and plain-language reasoning |

In sandbox mode, nothing above touches a real wallet — `status` comes back `"sandbox"` and no transaction is broadcast. Outside sandbox mode, DPX returns an `execution` object (router address, token, amount, quote ID) and the paying agent's own wallet independently signs and broadcasts `approve()` + `router.settle()` — DPX never holds funds or a private key at any point.

## Real proof: this exact flow, run for real

The cell output above runs in sandbox mode. Here's what the same two-call flow produced when run for real, on Base mainnet, as two genuinely separate agent processes with no shared code path — one $1 settlement, real funds, real on-chain confirmation.

**Invoice**
```json
{
  "invoiceId": "2c846308-9071-4313-9c1a-1f951d6d7be3",
  "amount": 1,
  "currency": "USD",
  "status": "OPEN"
}
```

**Settlement authorization**
```json
{
  "settlementId": "dpx_549e32f44119abdbf95558edea33b0ca",
  "status": "authorized",
  "grossAmount": 1,
  "feesTotal": 0.0164,
  "netAmount": 0.9836,
  "oracleStatus": "STABLE", "oracleScore": 77,
  "complianceScreen": {"status": "CLEAR", "amlScore": 24},
  "aiDecision": "EXECUTE", "aiConfidence": 0.98
}
```

**On-chain receipt**

| Step | Tx hash |
|---|---|
| `approve()` | [`0x962454379c1dca682b2f2cfff0f1d55833619d37fb4e197398ff6b450e3c14f1`](https://basescan.org/tx/0x962454379c1dca682b2f2cfff0f1d55833619d37fb4e197398ff6b450e3c14f1) |
| `router.settle()` | [`0xd12437bcc126b247ed0dc7551f2c3ef1397d4454ddffcfb51f8addbb60ac86d2`](https://basescan.org/tx/0xd12437bcc126b247ed0dc7551f2c3ef1397d4454ddffcfb51f8addbb60ac86d2) |

Verified independently on-chain: the receiver's USDC balance increased by the net amount, and the ESG redistribution contract's balance increased by exactly the ESG fee portion — in the same transaction, confirming the fee split and redistribution are genuinely atomic.